<a href="https://colab.research.google.com/github/narame7/UOS-FootballDataAnalytics-Tutorial/blob/main/%EC%B6%95%EA%B5%AC_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_2%EA%B0%95_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 통계 데이터 크롤링

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
import ScraperFC as sfc

ModuleNotFoundError: No module named 'ScraperFC'

## 필요한 라이브러리 import

In [ ]:
!pip install ScraperFC

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 kB 21.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.2/171.2 kB 12.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## [ClubElo](http://clubelo.com/)
- 팀들의 역대 Elo 점수를 알 수 있는 사이트.


In [ ]:
ce = sfc.ClubElo()
ce.scrape_team_on_date('Tottenham', '2023-08-13')

 ## [FBref](https://fbref.com/en/)
- 팀과 선수의 세부적인 기록을 가장 자세히 제공하는 사이트.

In [ ]:
fb = sfc.FBref()

In [ ]:
# valid_league = [
#     'Copa Libertadores', 'Champions League', 'Europa League', 'Europa Conference League',
#     'World Cup', 'Copa America', 'Euros', 'Big 5 combined', 'EPL',
#     'Ligue 1', 'Bundesliga', 'Serie A', 'La Liga', 'MLS', 'Brazilian Serie A',
#     'Eredivisie', 'Liga MX', 'Primeira Liga', 'Belgian Pro League', 'Argentina Liga Profesional',
#     'Saudi Pro League', 'Turkish Super Lig', 'EFL Championship', 'La Liga 2', '2. Bundesliga',
#     'Ligue 2', 'Serie B', 'Womens Champions League', 'Womens World Cup',
#     'Womens Euros', 'NWSL', 'A-League Women', 'WSL', 'D1 Feminine',
#     'Womens Bundesliga', 'Womens Serie A', 'Liga F', 'NWSL Challenge Cup', 'NWSL Fall Series'
# ]


### 리그 순위 불러오기

In [ ]:
league = fb.scrape_league_table('2025-2026', 'EPL')

In [ ]:
league[0]

### 경기 정보 불러오기

In [ ]:
links = fb.get_match_links('2025-2026', 'EPL') # 경기 링크 불러오기
match = fb.scrape_match(links[3]) # 불러온 링크 적용하기

In [ ]:
match

In [ ]:
match.loc[0,'Away Player Stats']['Summary'].head()

### 특정 경기의 슈팅 창출 순위 구하기

In [ ]:
def flatten_columns(df):
    df = df.copy()
    new_cols = []
    for col in df.columns.to_flat_index():
        upper, lower = col
        if "Unnamed" in str(upper):
            # 위쪽 이름이 없으면 아래쪽만 사용
            new_cols.append(lower)
        else:
            new_cols.append(f"{upper}_{lower}")


    df.columns = new_cols
    df = df[~df['Player'].str.contains("Players", na=False)].copy()
    return df

home_df = match.loc[0,'Home Player Stats']['Summary']
away_df = match.loc[0,'Away Player Stats']['Summary']
home = flatten_columns(home_df)
away = flatten_columns(away_df)

rename_map = {
    'Performance_Sh': 'Shots',
    'SCA_SCA': 'SCA',
    'SCA_GCA': 'GCA',
    'Performance_Min': 'Min',
    'Expected_xG': 'xG',
    'Passes_PrgP': 'Progressive Pass'
}
home = home.rename(columns=rename_map)
away = away.rename(columns=rename_map)

all_players = pd.concat([home, away], ignore_index=True)
all_players['SCA'] = pd.to_numeric(all_players['SCA'], errors='coerce')
all_players_sorted = all_players.sort_values('SCA', ascending=False)

keep_cols = [c for c in ['Player', 'Team', 'Pos', 'Age', 'Min','Shots', 'xG', 'Progressive Pass', 'SCA', 'GCA'] if c in all_players_sorted.columns]
all_players_sorted[keep_cols].head(15)

## [Understat](https://understat.com/)

- 자체 xG 모델과 여러 기록이 있는 통계 사이트

In [ ]:
us = sfc.Understat()

### 링크 불러오기

In [ ]:
us_match_links = us.get_match_links('2025/2026', 'EPL')
us_season_links = us.get_season_link('2025/2026', 'EPL')
us_team_links = us.get_team_links('2025/2026', 'EPL')

### 팀 순위 보기

In [ ]:
tables = us.scrape_league_tables('2024/2025', 'EPL')

In [ ]:
tables[0] # 전체 순위, 홈 경기 순위, 원정 경기 순위

### 팀 내 순위 보기

In [ ]:
team_data = us.scrape_all_teams_data("2025/2026", "EPL", as_df=True)

In [ ]:
team_data.keys()

In [ ]:
team_data['https://understat.com/team/Everton/2025'].keys()

In [ ]:
team_data['https://understat.com/team/Everton/2025']['players_data'].head()

### 경기 정보 보기

In [ ]:
us_match = us.scrape_match(us_match_links[0], as_df=False)

In [ ]:
us_match[0].keys() #home away 구분

In [ ]:
us_match_df = pd.DataFrame(us_match[0]['h']) # 첫번째 경기의 홈 팀 정보


In [ ]:
us_match_df.head()

### 여러 경기 보기

In [ ]:
us_matches = us.scrape_matches('2025/2026', 'EPL', as_df=False)

In [ ]:
us_matches.keys()

In [ ]:
pd.DataFrame(us_matches['https://understat.com/match/28778'])

### 시즌 누적 데이터 불러오기

In [ ]:
us_season_data = us.scrape_season_data('2025/2026', 'EPL')

In [ ]:
pd.DataFrame(us_season_data[2])

### 특정 팀의 경기 데이터 불러오기

In [ ]:
us_team_data = us.scrape_team_data(us_team_links[0], as_df=False) # 특정 팀(aston villa)의 팀 데이터 불러오기

In [ ]:
pd.DataFrame(us_team_data[2]) # 일정, 공격 진행 통계, 팀 내 순위

# 이벤트 데이터, 트래킹 데이터 수집